In [1]:
import pandas as pd

In [2]:
data = pd.read_csv('/Users/sudeepmungara/Documents/Personal_Projects/Flight-Delay-Classification/Artifacts/03_02_2025_00_15_21/data_ingestion/ingested/train.csv')

In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 161807 entries, 0 to 161806
Data columns (total 12 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   MONTH                      161807 non-null  int64  
 1   DAY_OF_MONTH               161807 non-null  int64  
 2   DAY_OF_WEEK                161807 non-null  int64  
 3   UNIQUE_CARRIER             161807 non-null  object 
 4   DISTANCE                   161807 non-null  float64
 5   ORIGIN_CITY_TIME_ZONE      161807 non-null  object 
 6   DEST_CITY_TIME_ZONE        161807 non-null  object 
 7   SCHD_ARR_TIME_UTC_MINUTES  161807 non-null  int64  
 8   EXPECTED_DURATION          161807 non-null  int64  
 9   NEW_ORIGIN                 161807 non-null  object 
 10  NEW_DEST                   161807 non-null  object 
 11  ARR_DELAY_CLS              161807 non-null  int64  
dtypes: float64(1), int64(6), object(5)
memory usage: 14.8+ MB


In [5]:
import yaml

In [6]:
# Create a list of dictionaries for all columns with their data types
columns_schema = [{col: str(dtype)} for col, dtype in data.dtypes.items()]

# Identify numerical columns (e.g., int and float types)
numerical_cols = data.select_dtypes(include=['number']).columns.tolist()

# Construct the overall schema dictionary matching your YAML structure
schema = {
    "columns": columns_schema,
    "numerical_columns": numerical_cols
}

In [7]:
with open("/Users/sudeepmungara/Documents/Personal_Projects/Flight-Delay-Classification/data_schema/schema.yaml", "w") as file:
    yaml.dump(schema, file, default_flow_style=False)

In [16]:
test_df = pd.read_csv('/home/sudeep/Development/Learning/Flight-Delay-Classification/Artifacts/03_03_2025_12_01_25/data_validation/validated/test.csv')
train_df = pd.read_csv('/home/sudeep/Development/Learning/Flight-Delay-Classification/Artifacts/03_03_2025_12_01_25/data_validation/validated/train.csv')
TARGET_COLUMN = 'ARR_DELAY_CLS'

In [17]:
train_data.head(1)

,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,UNIQUE_CARRIER,DISTANCE,ORIGIN_CITY_TIME_ZONE,DEST_CITY_TIME_ZONE,SCHD_ARR_TIME_UTC_MINUTES,EXPECTED_DURATION,NEW_ORIGIN,NEW_DEST,ARR_DELAY_CLS
0,7,1,6,DL,432.0,America/Indiana/Indianapolis,America/New_York,1181,74,Other,ATL,0


In [18]:
input_feature_train_df=train_df.drop(columns=[TARGET_COLUMN],axis=1)
target_feature_train_df = train_df[TARGET_COLUMN]

input_feature_test_df = test_df.drop(columns=[TARGET_COLUMN], axis=1)
target_feature_test_df = test_df[TARGET_COLUMN]

In [27]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler,OrdinalEncoder
import numpy as np

In [23]:
preprocess_numeric_features = ['DISTANCE','EXPECTED_DURATION']
preprocess_ordinal_features = ['MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK'] #Features with ordinal characterstics
preprocess_categorical_features = ['UNIQUE_CARRIER','ORIGIN_CITY_TIME_ZONE', 'DEST_CITY_TIME_ZONE','NEW_ORIGIN', 'NEW_DEST']# Features where one-hot encoding is required

In [24]:
preprocessor = ColumnTransformer(
transformers=[
    ('ord', OrdinalEncoder(),preprocess_ordinal_features),
    ('cat', OneHotEncoder(),preprocess_categorical_features ),
    ('num',MinMaxScaler(),preprocess_numeric_features),
])
pipeline = Pipeline(steps=[
('preprocessor', preprocessor)])

In [ ]:
preprocessor_object=pipeline.fit(input_feature_train_df)
transformed_input_train_feature = pipeline.transform(input_feature_train_df)
transformed_input_test_feature =pipeline.transform(input_feature_test_df)

In [29]:
print("Train features shape:", transformed_input_train_feature.shape)
print("Train target shape:", np.array(target_feature_train_df).shape)
print("Test features shape:", transformed_input_test_feature.shape)
print("Test target shape:", np.array(target_feature_test_df).shape)

Train features shape: (161807, 62)
Train target shape: (161807,)
Test features shape: (40452, 62)
Test target shape: (40452,)


In [36]:
train_arr = np.c_[transformed_input_train_feature.toarray(), np.array(target_feature_train_df).reshape(-1, 1) ]
test_arr = np.c_[ transformed_input_test_feature.toarray(), np.array(target_feature_test_df).reshape(-1, 1) ]

In [33]:
np.array(target_feature_train_df)

array([0, 0, 0, ..., 0, 0, 0])

In [32]:
np.array(target_feature_train_df).reshape(-1,1)

array([[0],
       [0],
       [0],
       ...,
       [0],
       [0],
       [0]])

In [38]:
train_arr.shape

(161807, 63)

In [39]:
test_arr.shape

(40452, 63)